# ЛР2 — синонимическое обезличивание персональных данных

**Основа:** `Lab2_run_all.ipynb`. Здесь оставлены только функции, необходимые для готовой программы. **Обучения, скачивания factRuEval и генерации обучающих данных нет.**

Для автоматического распознавания используется **уже обученная локальная RuBERT** из каталога `rubert_pii_fio_extra` рядом с ноутбуком (или измените `MODEL_DIR` ниже).

Обезличиваем только шесть атрибутов: `NAME`, `SURNAME`, `EMAIL`, `INN`, `PHONE`, `CARD`. На вход `anon(text, entities)` подаются координаты в формате ЛР1: один атрибут `['NAME', 0, 4]` или список `[['NAME', 0, 4], ...]`. Для автоматического режима используется `detect_pii(text)`.

**Порядок:** `Kernel → Restart Kernel and Run All Cells`; затем изменяйте текст и координаты в последней пользовательской ячейке. Её можно запускать повторно без обучения.

In [1]:
# Импорты и путь к уже сохранённой модели (ничего не обучаем)
import re
import string
from pathlib import Path
from datetime import date, datetime
from random import SystemRandom

import torch
import pymorphy3
from faker import Faker
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Папка с обученной RuBERT должна содержать config.json и веса.
# Если модель в другом месте, укажите полный путь, например:
# MODEL_DIR = Path(r"D:\projects\pii\rubert_pii_fio_extra")
MODEL_DIR = Path("rubert_pii_fio_extra").resolve()

print("Python-окружение использует PyTorch:", torch.__version__)
print("Устройство:", "CUDA" if torch.cuda.is_available() else "CPU")
print("Каталог модели:", MODEL_DIR)

Python-окружение использует PyTorch: 2.14.1+cpu
Устройство: CPU
Каталог модели: C:\Users\Vlad\1JupyterLab\CyberSec\rubert_pii_fio_extra


## 1. Распознавание структурированных данных

Регулярные выражения и проверки контрольных разрядов перенесены из исходного ноутбука без изменений. Дополнительные распознаваемые классы из ЛР1 при обезличивании игнорируются.

In [2]:
#Регулярки
STRUCTURED_TYPES = ("EMAIL","PHONE","INN","CARD","SNILS","BIRTH_DATE")
PATTERNS = {
    "EMAIL": re.compile( # [!a1_.@] [буквы+спецсимволы] [@] [буквы,цифры,дефисы] [.] [минимум две буквы-цифры] [!a1_+]
        r"(?<![\w.@])[\w.%+\-]+@(?:[A-Za-z0-9\-]+\.)+[A-Za-z]{2,}(?![\w@-])", re.I),
    "PHONE": re.compile(
        r"(?<![\w+])(?:\+7|8)[ \t-]?\(?[3489]\d{2}\)?"   # [!a1_+] [+7|8] [ |/t|-] [(] [3|4|8|9] [xx] [)]
        r"[ \t-]?\d{3}[ \t-]?\d{2}[ \t-]?\d{2}(?![\w-])" # [ |/t|-] [xxx] [ |/t|-] [xx] [ |/t|-] [xx] [!a1_+]
    ),                                                   # неподдерж вне-рф, и "79231116145" 
    "INN": re.compile(r"(?<!\d)\d{12}(?!\d)"),  #[!1][xxxxxxxxxxxx][!1]
    "CARD": re.compile(r"(?<!\d)\d(?:[ -]?\d){12,18}(?![\d-])"),  #visa 13,16,19, Mastercard 16, AMEX 15, MIR 16-19
    "SNILS": re.compile(r"(?<!\d)(?:\d{3}[- ]\d{3}[- ]\d{3}[- ]\d{2}|\d{11})(?!\d)"),  #[!1] [xxx] [ |-] [xxx] [ |-] [xxx] [ |-] [xx] [!1]
    "BIRTH_DATE": re.compile(r"(?<!\d)(?:\d{2}[./-]\d{2}[./-]\d{4}|\d{4}-\d{2}-\d{2})(?!\d)") #[!1] [xx] [./-] [xx] [./-] [xxxx]
}

In [3]:
#Контрольные суммы
#Удаляет все символы, кроме цифр.
def digits_only(value: str) -> str:
    return re.sub(r"\D", "", value)

#Проверка контрольной суммы алгоритмом Луна.
#1. Цифры проверяемой последовательности нумеруются справа налево.
#2. Цифры, оказавшиеся на нечётных местах, остаются без изменений.
#3. Цифры, стоящие на чётных местах, умножаются на 2.
#4. Если в результате такого умножения возникает число больше 9, 
    #оно заменяется суммой цифр получившегося произведения — однозначным числом, то есть цифрой.
#5. Все полученные в результате преобразования цифры складываются. Если сумма кратна 10, то исходные данные верны.
def valid_luhn(value: str) -> bool:
    number = digits_only(value)
    if (not number.isascii() or not number.isdigit() or not 13 <= len(number) <= 19):
        return False
    total = 0

    for index, char in enumerate(reversed(number)):
        digit = int(char)
        if index % 2:
            digit *= 2
            if digit > 9:
                digit -= 9
        total += digit
    return total % 10 == 0

#Проверка номера банковской карты.
def valid_card(value: str) -> bool:
    number = digits_only(value)

    return (
        13 <= len(number) <= 19  # ну вдруг
        and number[0] in "23456" # 2 мир, мастеркард, 3 amex, 4 visa, 5 mastercard, 6 discover, unionpay
        and len(set(number)) > 1
        and valid_luhn(number)
    )

#Проверка 12-значного ИНН физического лица.
def valid_inn(value: str) -> bool:
    if not re.fullmatch(r"[0-9]{12}", value):
        return False
    numbers = list(map(int, value))
    w11 = (7, 2, 4, 10, 3, 5, 9, 4, 6, 8)    #веса для расчёта контрольной суммы
    w12 = (3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8)
    c11 = sum(n * w for n, w in zip(numbers[:10], w11)) % 11 % 10
    c12 = sum(n * w for n, w in zip(numbers[:11], w12)) % 11 % 10
    return (c11, c12) == (numbers[10], numbers[11])

#Проверка контрольного числа СНИЛС.
def valid_snils(value: str) -> bool:
    number = digits_only(value)
    if (len(number) != 11 or not number.isascii() or not number.isdigit()):
        return False
    if int(number[:9]) <= 1001998:   # снилс менее 001-001-998 считается по другому, отметаем
        return False

    weighted_sum = sum(int(c) * (9 - i) for i, c in enumerate(number[:9]))
    control = weighted_sum % 101
    if control == 100:
        control = 0
    return control == int(number[-2:])

#Проверяет корректность календарной даты.
def valid_birth_date(value: str) -> bool:
    formats = ("%d.%m.%Y","%d/%m/%Y","%d-%m-%Y","%Y-%m-%d")
    for fmt in formats:
        try:
            parsed = datetime.strptime(value, fmt).date()
            return (
                date(1900, 1, 1)
                <= parsed
                <= date.today()
            )
        except ValueError:
            continue
    return False

In [4]:
#Определяем контекст даты рождения.
def birth_context(text: str, start: int, end: int) -> bool:
    before = text[max(0, start - 60):start]
    after = text[end:min(len(text), end + 20)]

    lead = re.search(
        r"(?:дата\s+рождения|д\.\s*р\.?|"
        r"родил[а-яё]*|день\s+рождения)"
        r"\s*[:\-]?\s*$",
        before,
        re.I
    )

    trail = re.match(
        r"\s*(?:г\.?\s*р\.?|года\s+рождения)"
        r"(?![а-яё\w])",
        after,
        re.I
    )

    return bool(lead or trail)

In [5]:
# Структурированные ПДн
def detect_structured(text: str) -> list[list]:
    if not isinstance(text, str):
        raise TypeError("text должен быть строкой")

    validators = {
        "INN": valid_inn,
        "CARD": valid_card,
        "SNILS": valid_snils,
        "BIRTH_DATE": valid_birth_date
    }

    priority = {
        "EMAIL": 6,
        "PHONE": 5,
        "CARD": 4,
        "INN": 3,
        "SNILS": 2,
        "BIRTH_DATE": 1
    }

    candidates = []
    for label, pattern in PATTERNS.items():
        for match in pattern.finditer(text):
            start, end = match.span()
            value = match.group()

            # Проверка контрольных разрядов и даты
            if label in validators:
                if not validators[label](value):
                    continue

            # Для СНИЛС без разделителей
            # требуем явное указание типа
            if label == "SNILS" and value.isdigit():
                preceding = text[max(0, start - 25):start]

                if not re.search(
                    r"снилс\s*[:№-]?\s*$",
                    preceding,
                    re.I
                ):
                    continue

            # Дата должна быть датой рождения
            if label == "BIRTH_DATE":
                if not birth_context(text, start, end):
                    continue

            # Проверка длины российского телефона
            if label == "PHONE":
                if len(digits_only(value)) != 11:
                    continue

            candidates.append((label, start, end))

    # Сортировка по приоритету
    candidates.sort(
        key=lambda x: (
            -priority[x[0]],
            -(x[2] - x[1]),
            x[1]
        )
    )

    # Исключение пересекающихся сущностей
    selected = []

    for label, start, end in candidates:
        overlap = any(
            start < e and s < end
            for _, s, e in selected
        )

        if not overlap:
            selected.append((label, start, end))

    # Сортировка по позиции в тексте
    selected.sort(key=lambda x: (x[1], x[2]))

    return [
        [label, start, end]
        for label, start, end in selected
    ]

## 2. Загрузка сохранённой RuBERT

Модель **только загружается** из локального каталога. Если веса отсутствуют, выдаётся понятная ошибка вместо автоматического обучения.

In [6]:
# Загрузка уже обученной модели из Lab2_run_all.ipynb (без Trainer/train)
_model_config = MODEL_DIR / "config.json"
_model_weights = (
    "model.safetensors", "pytorch_model.bin",
    "model.safetensors.index.json", "pytorch_model.bin.index.json"
)
if not _model_config.is_file() or not any((MODEL_DIR / f).is_file() for f in _model_weights):
    raise FileNotFoundError(
        f"Сохранённая RuBERT не найдена в {MODEL_DIR}. "
        "Положите ноутбук рядом с папкой rubert_pii_fio_extra "
        "или измените MODEL_DIR в первой ячейке."
    )

morph = pymorphy3.MorphAnalyzer()
new_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=True, local_files_only=True)
new_tokenizer.model_max_length = 128
new_model = AutoModelForTokenClassification.from_pretrained(str(MODEL_DIR), local_files_only=True)
new_ner = pipeline(
    "token-classification",
    model=new_model,
    tokenizer=new_tokenizer,
    aggregation_strategy="first",
    stride=32,
    device=0 if torch.cuda.is_available() else -1
)
print("RuBERT загружена с диска. Обучение не выполнялось.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RuBERT загружена с диска. Обучение не выполнялось.


In [7]:
# Распознавание только ФИО

# Обращения и титулы, не являющиеся именами
PERSON_TITLES = {
    "миссис", "мисс", "мистер",
    "господин", "госпожа",
    "мадам", "сэр"
}

def detect_person(text):
    entities = []

    for entity in new_ner(text):
        label = entity["entity_group"]
        start = entity["start"]
        end = entity["end"]

        value = text[start:end].strip()
        if not any(ch.isalpha() for ch in value):
            continue
        normalized = value.lower()
        # Отсекаем очевидные нарицательные существительные,
        # ошибочно распознанные как NAME или SURNAME.
        if (
            label in {"NAME", "SURNAME"}
            and re.fullmatch(r"[а-яё]+", value)
        ):
            parses = morph.parse(value)

            if (
                any("NOUN" in p.tag for p in parses)
                and not any(
                    "Name" in p.tag or "Surn" in p.tag
                    for p in parses
                )
            ):
                continue
        # Исключаем обращения из распознанных ФИО
        if normalized in PERSON_TITLES:
            continue

        # Отсекаем конкретное ложное срабатывание
        # Но разрешаем «Цыган» с заглавной буквы
        # как потенциальную фамилию
        if label == "NAME" and value == "цыган":
            continue

        entities.append([label, start, end])

    return entities


# Распознавание всех ПДн
def detect_pii(text):
    if not isinstance(text, str):
        raise TypeError("text должен быть строкой")

    result = detect_structured(text)

    for label, start, end in detect_person(text):
        if not any(start < e and s < end for _, s, e in result):
            result.append([label, start, end])

    return sorted(result, key=lambda x: x[1])

## 3. Алгоритмы синонимического обезличивания

Базовые функции из исходного ноутбука сохранены: морфологический анализ `pymorphy3`, генерация с `Faker`/`SystemRandom`, сохранение префиксов/форматирования и контрольных сумм.

Для редких и иностранных ФИО возможны неоднозначности рода и падежа; они не гарантированно разрешаются автоматически. При невозможности анализа функция выдаёт ошибку, а не обещает корректную замену.

In [8]:
# ЛР2: необходимые объекты без запуска обучения RuBERT
import re
import string
import pymorphy3
from faker import Faker
from random import SystemRandom

PERSON_TAGS = {"NAME": "Name", "SURNAME": "Surn"}
morph = pymorphy3.MorphAnalyzer()
anon_fake = Faker("ru_RU")
anon_rng = SystemRandom()

In [9]:
# Фамилии, не изменяющиеся по падежам
INDECLINABLE_SURNAMES = [
    "Гюго", "Дюма", "Верди", "Моро", "Россини", "Моне"
]

INITIAL_LETTERS = "АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЭЮЯ"


def restore_person_case(original, replacement):
    if original.isupper():
        return replacement.upper()
    if original.islower():
        return replacement.lower()
    if original.istitle():
        return replacement.capitalize()
    return replacement


def replace_person_base(value, label, replacements, left_context=""):
    if label not in PERSON_TAGS or not value:
        raise ValueError(f"Некорректный атрибут: {label}, {value!r}")

    # 1. Инициалы: М -> К, М. -> К.
    initial = re.fullmatch(r"([А-Яа-яЁё])(\.?)", value)

    if initial:
        letter, dot = initial.groups()
        key = (label, "INITIAL", letter.upper())

        if key not in replacements:
            replacements[key] = anon_rng.choice([
                c for c in INITIAL_LETTERS
                if c != letter.upper()
            ])

        return restore_person_case(
            letter, replacements[key]
        ) + dot

    # 2. Обычный морфологический анализ
    tag = PERSON_TAGS[label]
    parses = [
        p for p in morph.parse(value)
        if tag in p.tag
        and p.tag.gender in ("masc", "femn")
    ]

    # Запасной анализ редких имён:
    # Робера, Фауста, Эдисона и т.п.
    if not parses and label == "NAME":
        parses = [
            p for p in morph.parse(value)
            if "NOUN" in p.tag
            and "anim" in p.tag
            and p.tag.gender in ("masc", "femn")
            and p.tag.case is not None
        ]

    if parses:
        original = parses[0]
        gender = original.tag.gender
        case = original.tag.case
        number = original.tag.number
        normal_form = original.normal_form

    elif label == "SURNAME":
        # 3. Неизвестная склоняемая фамилия:
        # Брохесом -> предположительно мужской, тв. падеж
        if value.lower().endswith(("ом", "ем")):
            gender = "masc"
            case = "ablt"
            number = "sing"
            normal_form = value[:-2].lower()

        else:
            # 4. Неоднозначная или несклоняемая фамилия.
            # Используем другую несклоняемую фамилию:
            # Мураками -> Верди, Моне -> Гюго.
            key = ("SURNAME", "INDECL", value.lower())

            if key not in replacements:
                candidates = [
                    s for s in INDECLINABLE_SURNAMES
                    if s.lower() != value.lower()
                ]
                replacements[key] = anon_rng.choice(candidates)

            return restore_person_case(
                value, replacements[key]
            )


    elif label == "NAME":
        lower = value.lower()

        if lower == "лизка":
            gender = "femn"
            case = "nomn"
            number = "sing"
            normal_form = "лизка"

        elif (
            lower == "мая"
            and re.search(r"\bмальчика\s+$", left_context.lower())
        ):
            gender = "masc"
            case = "gent"
            number = "sing"
            normal_form = "май"

        elif lower.endswith(("ом", "ем")):
            # Орсоном -> Орсон, творительный падеж
            gender = "masc"
            case = "ablt"
            number = "sing"
            normal_form = lower[:-2]

        elif lower.endswith(("ей", "ий", "й")):
            # Кащей -> Кащей, предполагаемый именительный
            gender = "masc"
            case = "nomn"
            number = "sing"
            normal_form = lower

        else:
            raise ValueError(
                f"Неизвестная форма NAME: {value}"
            )

    # 5. Согласование повторов по нормальной форме
    key = (label, normal_form, gender)
    grammemes = {g for g in (case, number) if g}

    if key not in replacements:
        sex = "male" if gender == "masc" else "female"
        method = "first_name" if label == "NAME" else "last_name"

        for _ in range(100):
            candidate = getattr(
                anon_fake, f"{method}_{sex}"
            )()

            options = [
                p for p in morph.parse(candidate)
                if tag in p.tag and p.tag.gender == gender
            ]

            if not options:
                continue

            candidate_parse = options[0]

            if candidate_parse.normal_form == normal_form:
                continue

            if grammemes and candidate_parse.inflect(grammemes) is None:
                continue

            replacements[key] = candidate_parse
            break
        else:
            raise ValueError(
                f"Не удалось подобрать замену: {value}"
            )

    # 6. Склонение выбранной замены
    replacement = replacements[key]

    inflected = (
        replacement.inflect(grammemes)
        if grammemes else replacement
    )

    if inflected is None:
        raise ValueError(f"Ошибка склонения: {value}")

    return restore_person_case(value, inflected.word)

In [10]:
# Расширение базового алгоритма: уменьшительные имена и латиница

latin_fake = Faker("en_US")

LATIN_FEMALE = {"bonnie"}
LATIN_MALE = {"george", "walt"}


def replace_person(value, label, replacements, left_context=""):
    # Неоднозначное по роду имя: Саша
    if label == "NAME" and value.casefold() == "саша":
        key = ("NAME", "NICKNAME", "саша")

        if key not in replacements:
            # Женя и Валя могут быть мужскими и женскими именами
            replacements[key] = anon_rng.choice(["Женя", "Валя"])

        return restore_person_case(value, replacements[key])

    # Латинский инициал: H -> R
    if label == "NAME" and re.fullmatch(r"[A-Za-z]", value):
        key = ("NAME", "LATIN_INITIAL", value.upper())

        if key not in replacements:
            letters = string.ascii_uppercase.replace(value.upper(), "")
            replacements[key] = anon_rng.choice(letters)

        return restore_person_case(value, replacements[key])

    # Латинские имена: George -> Michael
    if label == "NAME" and re.fullmatch(r"[A-Za-z]+", value):
        key = ("NAME", "LATIN", value.casefold())

        if key not in replacements:
            name = value.casefold()

            if name in LATIN_FEMALE:
                generate = latin_fake.first_name_female
            elif name in LATIN_MALE:
                generate = latin_fake.first_name_male
            else:
                # Пол неизвестен: нейтральные варианты
                generate = lambda: anon_rng.choice(
                    ["Alex", "Taylor", "Jordan", "Casey"]
                )

            for _ in range(100):
                candidate = generate()

                if candidate.casefold() != name:
                    replacements[key] = candidate
                    break
            else:
                raise ValueError(f"Не удалось заменить NAME: {value}")

        return restore_person_case(value, replacements[key])

    # Все обычные кириллические имена и фамилии
    return replace_person_base(
        value, label, replacements, left_context
    )

In [11]:
import re
import string
from random import SystemRandom

anon_rng = SystemRandom()


def random_other(char, alphabet):
    """Случайный символ, обязательно отличный от исходного."""
    return anon_rng.choice(alphabet.replace(char, ""))


def replace_email(value):
    """Меняет часть до @, сохраняя домен и оформление."""
    parts = value.split("@")

    if len(parts) != 2:
        raise ValueError("Некорректный EMAIL")

    local, domain = parts

    # Проверка некорректного расположения точек и дефисов
    if (
        local.startswith(".")
        or local.endswith(".")
        or ".." in local
        or ".." in domain
        or any(
            part.startswith("-") or part.endswith("-")
            for part in domain.split(".")
        )
    ):
        raise ValueError(f"Некорректный EMAIL: {value}")

    if not re.fullmatch(r"[A-Za-z0-9._%+-]+", local):
        raise ValueError("Некорректная локальная часть EMAIL")

    if not re.fullmatch(
        r"[A-Za-zА-Яа-яЁё0-9-]+(?:\.[A-Za-zА-Яа-яЁё0-9-]+)+",
        domain
    ):
        raise ValueError("Некорректный домен EMAIL")

    result = []

    for char in local:
        if char in string.ascii_lowercase:
            result.append(random_other(char, string.ascii_lowercase))
        elif char in string.ascii_uppercase:
            result.append(random_other(char, string.ascii_uppercase))
        elif char in string.digits:
            result.append(random_other(char, string.digits))
        else:
            result.append(char)

    return "".join(result) + "@" + domain


def replace_phone(value):
    """Сохраняет первые 4 цифры и все нецифровые символы."""
    digits = [c for c in value if c in string.digits]

    if not 10 <= len(digits) <= 15:
        raise ValueError("Некорректная длина PHONE")

    if any(c not in string.digits + "+-(). \t" for c in value):
        raise ValueError("Недопустимые символы в PHONE")

    result = []
    digit_index = 0

    for char in value:
        if char in string.digits:
            if digit_index < 4:
                result.append(char)
            else:
                result.append(random_other(char, string.digits))
            digit_index += 1
        else:
            result.append(char)

    return "".join(result)

In [12]:
def replace_inn(value):
    """Сохраняет первые 4 цифры и пересчитывает контрольные разряды."""

    if not valid_inn(value):
        raise ValueError(f"Некорректный ИНН: {value}")

    w11 = (7, 2, 4, 10, 3, 5, 9, 4, 6, 8)
    w12 = (3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8)

    while True:
        first10 = value[:4] + "".join(
            str(anon_rng.randrange(10)) for _ in range(6)
        )

        numbers = list(map(int, first10))

        c11 = sum(
            n * w for n, w in zip(numbers, w11)
        ) % 11 % 10

        c12 = sum(
            n * w for n, w in zip(numbers + [c11], w12)
        ) % 11 % 10

        result = first10 + str(c11) + str(c12)

        if result != value:
            return result

In [13]:
def replace_card(value):
    """Сохраняет первые 6 и последние 4 цифры, формат и Luhn."""

    if (
        not re.fullmatch(r"[0-9]+(?:[ \t-][0-9]+)*", value)
        or not valid_card(value)
    ):
        raise ValueError(f"Некорректная банковская карта: {value}")

    original = digits_only(value)

    for _ in range(1000):
        middle = "".join(
            str(anon_rng.randrange(10))
            for _ in range(len(original) - 10)
        )

        candidate = original[:6] + middle + original[-4:]

        if candidate != original and valid_luhn(candidate):
            digits_iter = iter(candidate)

            return "".join(
                next(digits_iter) if char in "0123456789" else char
                for char in value
            )

    raise RuntimeError("Не удалось сгенерировать валидную карту")

In [14]:
ANON_TYPES = {"NAME", "SURNAME", "EMAIL", "INN", "PHONE", "CARD"}


def restore_digits(original, new_digits):
    """Возвращает новые цифры в исходное оформление."""
    digits = iter(new_digits)
    return "".join(
        next(digits) if ch in "0123456789" else ch
        for ch in original
    )


def anon(text, entities):
    """Синонимически обезличивает ПДн по координатам из ЛР1."""

    if not isinstance(text, str):
        raise TypeError("Текст должен быть строкой")

    # Допускаем как один атрибут, так и список атрибутов
    if isinstance(entities, (list, tuple)) and len(entities) == 3 and isinstance(entities[0], str):
        entities = [entities]

    spans = []

    for entity in entities:
        if len(entity) != 3:
            raise ValueError(f"Некорректная сущность: {entity}")

        label, start, end = entity

        if not (isinstance(start, int) and isinstance(end, int)
                and 0 <= start < end <= len(text)):
            raise ValueError(f"Некорректные координаты: {entity}")

        if label in ANON_TYPES:
            spans.append((label, start, end))

    spans.sort(key=lambda x: x[1])

    for left, right in zip(spans, spans[1:]):
        if left[2] > right[1]:
            raise ValueError("Обнаружены пересекающиеся сущности")

    person_cache = {}
    structured_cache = {}
    result = text

    for label, start, end in reversed(spans):
        original = text[start:end]

        if label in ("NAME", "SURNAME"):
            replacement = replace_person(
                original,
                label,
                person_cache,
                left_context=text[max(0, start - 80):start]
            )

        elif label == "EMAIL":
            key = (label, original)

            if key not in structured_cache:
                structured_cache[key] = replace_email(original)

            replacement = structured_cache[key]

        else:
            # Нормализуем числовой атрибут
            normalized = digits_only(original)
            key = (label, normalized)

            # Проверяем каждое входное значение
            if label == "INN" and not valid_inn(original):
                raise ValueError(f"Некорректный ИНН: {original}")

            if label == "CARD" and (
                not re.fullmatch(r"[0-9]+(?:[ \t-][0-9]+)*", original)
                or not valid_card(original)
            ):
                raise ValueError(f"Некорректная карта: {original}")

            if label == "PHONE":
                if (
                    not 10 <= len(normalized) <= 15
                    or any(c not in "0123456789+-(). \t" for c in original)
                ):
                    raise ValueError(f"Некорректный телефон: {original}")

            if key not in structured_cache:
                generators = {
                    "INN": replace_inn,
                    "PHONE": replace_phone,
                    "CARD": replace_card
                }

                generated = generators[label](original)
                structured_cache[key] = digits_only(generated)

            replacement = restore_digits(
                original, structured_cache[key]
            )

        if replacement == original:
            raise ValueError(
                f"Значение не изменилось: {label} = {original}"
            )

        # Заменяем справа налево
        result = result[:start] + replacement + result[end:]

    return result

## 4. Автоматические проверки требований

Проверяем вход и выход, случайность, ИНН, телефон, карту и неожиданные значения. **RuBERT не обучается.**

In [15]:
text = (
    "Иванов Иван написал с ivan.petrov1988@gmail.com, "
    "указал ИНН 770698959438, телефон +7 (999) 123-45-67 "
    "и карту 4561 2612 1234 5467. "
    "Позднее с Ивановым Иваном связались повторно."
)

samples = [
    ("SURNAME", "Иванов"),
    ("NAME", "Иван"),
    ("EMAIL", "ivan.petrov1988@gmail.com"),
    ("INN", "770698959438"),
    ("PHONE", "+7 (999) 123-45-67"),
    ("CARD", "4561 2612 1234 5467"),
    ("SURNAME", "Ивановым"),
    ("NAME", "Иваном")
]

entities = []

for label, value in samples:
    match = re.search(
        rf"(?<!\w){re.escape(value)}(?!\w)", text
    )
    assert match is not None, f"Не найдено: {value}"
    entities.append([label, match.start(), match.end()])

first = anon(text, entities)
second = anon(text, entities)

print("ИСХОДНЫЙ ТЕКСТ:")
print(text)

print("\nОБЕЗЛИЧЕННЫЙ ТЕКСТ:")
print(first)

print("\nПОВТОРНЫЙ ЗАПУСК:")
print(second)

assert first != text
assert first != second

print("\nПроверка пройдена.")

ИСХОДНЫЙ ТЕКСТ:
Иванов Иван написал с ivan.petrov1988@gmail.com, указал ИНН 770698959438, телефон +7 (999) 123-45-67 и карту 4561 2612 1234 5467. Позднее с Ивановым Иваном связались повторно.

ОБЕЗЛИЧЕННЫЙ ТЕКСТ:
Степанов Остромир написал с lbtc.hbithy8737@gmail.com, указал ИНН 770664055410, телефон +7 (999) 777-67-72 и карту 4561 2618 1089 5467. Позднее с Степановым Остромиром связались повторно.

ПОВТОРНЫЙ ЗАПУСК:
Большаков Потап написал с dspk.queptb8651@gmail.com, указал ИНН 770633842696, телефон +7 (999) 981-50-32 и карту 4561 2683 0149 5467. Позднее с Большаковым Потапом связались повторно.

Проверка пройдена.


In [16]:
# Проверки ЛР2 без повторного обучения модели

# 1. Формат входа и выхода
email = "ivan.petrov@gmail.com"

single = anon(email, ["EMAIL", 0, len(email)])
multiple = anon(email, [["EMAIL", 0, len(email)]])

assert isinstance(single, str)
assert single != email
assert multiple != email
assert single.endswith("@gmail.com")
assert multiple.endswith("@gmail.com")
assert anon("Обычный текст.", []) == "Обычный текст."

print("✓ Вход и выход anon()")

# 2. Случайность при повторных вызовах
for label, value in [
    ("EMAIL", email),
    ("INN", "770698959438"),
    ("PHONE", "+7 (999) 123-45-67"),
    ("CARD", "4561 2612 1234 5467")
]:
    results = {
        anon(value, [label, 0, len(value)])
        for _ in range(20)
    }
    assert len(results) > 1, label

print("✓ Случайность всех структурированных атрибутов")

# 3. Контрольные разряды и формат
for _ in range(500):
    inn = replace_inn("770698959438")
    card = replace_card("4561-2612-1234-5467")
    phone = replace_phone("+7 (999) 123-45-67")

    assert inn.startswith("7706") and valid_inn(inn)

    assert valid_card(card)
    assert digits_only(card)[:6] == "456126"
    assert digits_only(card)[-4:] == "5467"
    assert re.fullmatch(r"\d{4}-\d{4}-\d{4}-\d{4}", card)

    assert digits_only(phone)[:4] == "7999"
    assert re.fullmatch(r"\+7 \(999\) \d{3}-\d{2}-\d{2}", phone)

print("✓ 500 повторных проверок INN, CARD, PHONE")

# 4. Реакция на некорректные значения
bad_cases = [
    (replace_email, "bob..a@gmail.com"),
    (replace_email, ".foo@gmail.com"),
    (replace_email, "ivan@a..com"),
    (replace_inn, "770698959430"),
    (replace_card, "4561 2612 1234 5469"),
    (replace_phone, "abc123"),
]

for func, value in bad_cases:
    try:
        func(value)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Не отклонено: {value}")

# 5. Ошибки координат
try:
    anon("Иван", [["NAME", 0, 100]])
except ValueError:
    pass
else:
    raise AssertionError("Не отклонены неверные координаты")

print("✓ Некорректные значения и координаты")
print("\nПроверки структурированных атрибутов завершены.")

✓ Вход и выход anon()
✓ Случайность всех структурированных атрибутов
✓ 500 повторных проверок INN, CARD, PHONE
✓ Некорректные значения и координаты

Проверки структурированных атрибутов завершены.


## 5. Ввод своего текста — основная ячейка

Измените `TEXT` и `ENTITIES` и нажмите `Shift+Enter`.

- **Ручная разметка из ЛР1:** `ENTITIES = ['NAME', 0, 4]` или `[['NAME', 0, 4], ['SURNAME', 5, 11]]`.
- **Автоматическое распознавание сохранённой RuBERT + правил:** установите `ENTITIES = None`.

Координаты — границы среза `TEXT[start:end]`, последний индекс не включается. При изменении текста обязательно обновите ручные координаты.

In [17]:
# МЕНЯЙТЕ ТОЛЬКО ЭТИ ДВЕ СТРОКИ ДЛЯ СВОЕГО ПРИМЕРА
TEXT = " Бедарев получил кредит"
ENTITIES = ["NAME", 0, 8]  # или None для автоматического распознавания

if ENTITIES is None:
    selected_entities = [
        e for e in detect_pii(TEXT)
        if e[0] in ANON_TYPES
    ]
    print("Режим: автоматический (RuBERT + правила)")
else:
    selected_entities = ENTITIES
    print("Режим: ручная разметка из ЛР1")

items = ([selected_entities] if isinstance(selected_entities, (list, tuple))
         and len(selected_entities) == 3 and isinstance(selected_entities[0], str)
         else selected_entities)
print("Найденные/заданные ПДн:", [
    (label, TEXT[start:end], start, end) for label, start, end in items
])
print("\nБЫЛО:", TEXT)
print("СТАЛО:", anon(TEXT, selected_entities))

Режим: ручная разметка из ЛР1
Найденные/заданные ПДн: [('NAME', ' Бедарев', 0, 8)]

БЫЛО:  Бедарев получил кредит
СТАЛО: Родион получил кредит


### Пример автоматической обработки

Здесь не нужно заполнять индексы самостоятельно: `detect_pii()` использует сохранённую RuBERT и алгоритмические правила. Этот пример также выполнится при `Run All`.

In [18]:
auto_text = "Иванов Иван, телефон +7 (999) 123-45-67."
auto_entities = [e for e in detect_pii(auto_text) if e[0] in ANON_TYPES]
print("Автоматически найдены:", [(k, auto_text[s:e]) for k,s,e in auto_entities])
print("Исходный текст:", auto_text)
print("Обезличенный:", anon(auto_text, auto_entities))

Автоматически найдены: [('SURNAME', 'Иванов'), ('NAME', 'Иван'), ('PHONE', '+7 (999) 123-45-67')]
Исходный текст: Иванов Иван, телефон +7 (999) 123-45-67.
Обезличенный: Сидоров Филипп, телефон +7 (999) 787-66-26.
